## Install libraries

In [1]:
!pip install -q youtube-transcript-api langchain-community \
              faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install -q sentence-transformers google-generativeai

In [3]:
!pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 1.5 MB/s eta 0:00:00


In [4]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai
from google.colab import userdata

from google.colab import userdata
gemini_api_key = userdata.get('GEMINI_API_KEY')
hugging_face_key = userdata.get("HF_TOKEN")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Step 1a - Indexing (Document Ingestion)

In [6]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter

video_id = "1IxG7ywSNXk"

# Fetch transcript
ytt_api = YouTubeTranscriptApi()

fetched_transcript = ytt_api.fetch(video_id)

# Convert transcript object into plain text
transcript_text = " ".join(
    snippet.text for snippet in fetched_transcript
)

In [7]:
transcript_text

'[Music] So Nepai, you\'re the CEO of Alphabet and Google. Welcome back to the coder. Well, good to be back. Feels like a nice tradition post IO to be talking to you. So good to be back. I think this is the third year we\'ve done this after IO. I\'m excited. Thank you for keeping the tradition alive. Lots to talk about. You announced a lot of things yesterday during the keynote. Uh there\'s AI mode rolling out for US users. Big updates to Gemini. There\'s V3 and Imagine the generators. You solved Pokemon with robots, which is very exciting. You know, my takeaway yesterday was that Google feels very confident now. There\'s there\'s a a real confidence about the technology coming to life and the products. A lot of things are shipping imminently. What\'s the one piece that gave you that confidence? Is it just the volume of things that are shipping? Is it one technology that clicked into being ready for consumers? Where where is it coming from? Look, I think it comes from the depth and bre

## Step 1b - Indexing (Text Splitting)

In [8]:
# Create splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Split text into chunks
chunks = splitter.create_documents([transcript_text])

# Print first chunk
print(chunks[0].page_content)

[Music] So Nepai, you're the CEO of Alphabet and Google. Welcome back to the coder. Well, good to be back. Feels like a nice tradition post IO to be talking to you. So good to be back. I think this is the third year we've done this after IO. I'm excited. Thank you for keeping the tradition alive. Lots to talk about. You announced a lot of things yesterday during the keynote. Uh there's AI mode rolling out for US users. Big updates to Gemini. There's V3 and Imagine the generators. You solved Pokemon with robots, which is very exciting. You know, my takeaway yesterday was that Google feels very confident now. There's there's a a real confidence about the technology coming to life and the products. A lot of things are shipping imminently. What's the one piece that gave you that confidence? Is it just the volume of things that are shipping? Is it one technology that clicked into being ready for consumers? Where where is it coming from? Look, I think it comes from the depth and breadth of


In [9]:
len(chunks)

54

In [10]:
chunks[50]

Document(metadata={}, page_content="AI overviews, I think, you know, people were adversely querying to find errors and the error rate was 1 in 7 million for adversal queries and so but that was a big but that's the bar. We've always operated as a company and so I think to me nothing is nothing has changed like you know Google operates under a very high bar that's the bar we strive to meet and and you know our search page results are there for everyone to see with that comes that natural uh accountability and you know we have to constantly earn people's trust so that's how I would approach it what do you think the marker is for the next phase of the platform shift after this one we we open by talking about we're in a second phase what's the marker for the final phase or the third phase of the platform shift you mean of the AI platform. What what are you looking for as the next marker? Oh, you know, look, I think, you know, the the the real thing about AI, which I think why I've always c

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [11]:
!pip install faiss-cpu

In [12]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    task_type="RETRIEVAL_DOCUMENT",
    google_api_key=gemini_api_key
)

vector_store = FAISS.from_documents(chunks, embeddings)

In [13]:
vector_store.index_to_docstore_id

{0: 'b9105634-5384-4293-8485-41b3a3b3cbd1',
 1: 'f74bbc4e-7a3c-4e87-8302-14e49c601b30',
 2: 'b5351d5c-4c5d-4763-87c9-c9c8465b77e1',
 3: 'a6c3454d-3a7c-4442-bc2f-3f98fb029fe6',
 4: '2128f4b6-7e72-4384-bd6e-2277e02c8618',
 5: '2b47ed9c-d450-4a7d-ac9d-571a1d167dc0',
 6: 'ea55f57b-b41d-4ae8-aa98-38d70fd0beef',
 7: '71807e1f-71d3-4777-8798-93926966a20d',
 8: '4b6b92e5-a4f3-4216-a871-818d856bcc43',
 9: 'bf0d164f-39c5-4c3f-9d9a-bdabad0f7202',
 10: 'ad00ca7b-779b-4ebd-a8ed-cbe0b4324567',
 11: 'b9ea5293-dfee-4d31-b810-d3a73e3de011',
 12: 'efac530d-d43b-4324-8d40-5914bb78c77a',
 13: 'adcb83ca-d614-4efd-bc3f-79b4cfce5b17',
 14: 'a9660b92-0f63-49c7-a182-56b969a29984',
 15: '3fcd3928-d000-43cc-a752-605ea4e0becb',
 16: '26371293-9319-4921-a395-f81c50f0b095',
 17: '79f5100b-f9f6-452c-815e-220547379d53',
 18: '67970585-2690-4c71-bd28-47ed4f1b636f',
 19: '0c61bbbe-0b52-44b0-a3f5-655f8edad883',
 20: 'ef28bddf-c547-4b05-85ed-9fa264ebf98c',
 21: 'c53ccbd2-1b90-4405-886b-9d08723214f2',
 22: '47db8756-37a9-

In [33]:
vector_store.get_by_ids(['59f55f34-1d75-4648-98d4-ba3b03b677d0'])

[Document(id='59f55f34-1d75-4648-98d4-ba3b03b677d0', metadata={}, page_content="around a robot but I'm talking more uh general purpose robot and you know and when AI creates that magical moment with robotics I think that'll be a big platform shift as well. Yeah I'm looking forward to it next year we're going to do this with glasses and robots. It's going to be great. We'll we'll give it a shot. Thank you so much. All right. Thanks. I appreciate it. Are these setups getting more and more stuff? I just feel like I have to say something more important just for this time.")]

## Step 2 - Retrieval

In [15]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [16]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7a310d1bc530>, search_kwargs={'k': 4})

In [34]:
retriever.invoke('Role of AI Agents in future')

[Document(id='a8fb60de-2002-43d7-befc-d9e31d85e1aa', metadata={}, page_content="would think about how to make that process more efficient. Today you're running a restaurant people are coming dining and eating and people are ordering takeout and delivery. Obviously for for you to service the takeout you would think about it different than all the tables and the clothing and the furniture and the like you know so but both are important to you. You could be a restaurant which decides I'm not going to participate in the takeout business. I'm only going to focus on uh on on on the dining experience. You're going to have some people vice versa. I'm going to say I'm going to go all in on this and run a different uh experience. So to your question on agents right I think think of agents as uh you know a new powerful format. I do think it'll happen in enterprises faster than consumer. Mhm. Because in the context of an enterprise you have a CIO who's able to go and say I really don't know why th

## Step 3 - Augmentation

In [51]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.2, google_api_key = gemini_api_key)

In [52]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [53]:
question = "is the topic of AI Agents discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [54]:
retrieved_docs

[Document(id='0e666504-bd1a-4580-888f-51745b6accd7', metadata={}, page_content="of databases for various agents to go and ask questions to and return those answers. And I I've been thinking about this in the context of services like Uber and Door Dash and Airbnb. Why would they want to participate in that and be abstracted away for agents to use the services they've spent a lot of time and money building? Two things though. First, there's a lot to unpack. It's a fascinating topic. Um the web is a series of databases, etc. We build a UI on top of it for for all of us to consume. This is exactly what I wanted is the web is a series of databases. It is like you know but I think I think I I listened to the uh te surrogate conversation yesterday I enjoyed it. I think he's saying for a agent first web like so you know for a web which is interacting with agents you would you would think about how to make that process more efficient. Today you're running a restaurant people are coming dining a

In [55]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"of databases for various agents to go and ask questions to and return those answers. And I I've been thinking about this in the context of services like Uber and Door Dash and Airbnb. Why would they want to participate in that and be abstracted away for agents to use the services they've spent a lot of time and money building? Two things though. First, there's a lot to unpack. It's a fascinating topic. Um the web is a series of databases, etc. We build a UI on top of it for for all of us to consume. This is exactly what I wanted is the web is a series of databases. It is like you know but I think I think I I listened to the uh te surrogate conversation yesterday I enjoyed it. I think he's saying for a agent first web like so you know for a web which is interacting with agents you would you would think about how to make that process more efficient. Today you're running a restaurant people are coming dining and eating and people are ordering takeout and delivery. Obviously for for you\n

In [56]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [57]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      of databases for various agents to go and ask questions to and return those answers. And I I've been thinking about this in the context of services like Uber and Door Dash and Airbnb. Why would they want to participate in that and be abstracted away for agents to use the services they've spent a lot of time and money building? Two things though. First, there's a lot to unpack. It's a fascinating topic. Um the web is a series of databases, etc. We build a UI on top of it for for all of us to consume. This is exactly what I wanted is the web is a series of databases. It is like you know but I think I think I I listened to the uh te surrogate conversation yesterday I enjoyed it. I think he's saying for a agent first web like so you know for a web which is interacting with agents you would you would th

## Step 4 - Generation

In [58]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of AI Agents is discussed in the video. The discussion revolves around the idea of an "agent-first web" where the web might transform into a series of databases that agents can query for information. This could lead to a more efficient process for agents to interact with services. The speaker also considers how services like Uber, Door Dash, and Airbnb might participate in such a system and notes that this shift might happen faster in enterprises than in consumer applications due to the ability of a CIO to mandate integration. The potential impact of AI on robotics is also mentioned as a future platform shift.


## Building a Chain

In [59]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [60]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [61]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [69]:
parallel_chain.invoke('What is AI Agents')

{'context': 'of databases for various agents to go and ask questions to and return those answers. And I I\'ve been thinking about this in the context of services like Uber and Door Dash and Airbnb. Why would they want to participate in that and be abstracted away for agents to use the services they\'ve spent a lot of time and money building? Two things though. First, there\'s a lot to unpack. It\'s a fascinating topic. Um the web is a series of databases, etc. We build a UI on top of it for for all of us to consume. This is exactly what I wanted is the web is a series of databases. It is like you know but I think I think I I listened to the uh te surrogate conversation yesterday I enjoyed it. I think he\'s saying for a agent first web like so you know for a web which is interacting with agents you would you would think about how to make that process more efficient. Today you\'re running a restaurant people are coming dining and eating and people are ordering takeout and delivery. Obvio

In [70]:
parser = StrOutputParser()

In [71]:
main_chain = parallel_chain | prompt | llm | parser

In [76]:
main_chain.invoke('Future of AI Agents')

'The future of AI agents is a fascinating topic. It\'s thought that agents will be adopted faster in enterprises than by consumers. In an enterprise context, a CIO can mandate that different systems communicate with each other, potentially reducing the need to purchase more databases for agents to query.\n\nFor consumer-facing services like Uber, DoorDash, and Airbnb, there\'s a question of whether they would want to participate and be abstracted away by agents using their services, which they\'ve invested heavily in building. However, it\'s also possible that these companies will see value in participating.\n\nSeveral models could emerge for how consumers interact with agents. One possibility is that consumers pay a subscription for agents, and the agents then revenue share back.\n\nUltimately, it\'s tough to predict precisely how this will unfold. However, the speaker believes that if agents are removing friction and improving user experience, it\'s difficult to bet against them in t